In [ ]:
from huggingface_hub import login
login(new_session=True)

In [ ]:
# 1. TensorRT-LLM 및 관련 라이브러리 설치
!pip install tensorrt_llm -U --pre --extra-index-url https://pypi.nvidia.com
!pip install --upgrade transformers # 최신 모델 지원을 위해 필수

# 2. TensorRT-LLM 예제 코드 다운로드 (스크립트 활용을 위해 필요)
!git clone https://github.com/NVIDIA/TensorRT-LLM.git
!cd TensorRT-LLM && git submodule update --init --recursive

# 3. 필요한 추가 의존성 설치
!pip install -r TensorRT-LLM/examples/llama/requirements.txt

In [ ]:
# 모델 다운로드 (git-lfs 활용)
!git lfs install
!git clone https://huggingface.co/TinyLlama/TinyLlama-1.1B-Chat-v1.0 tmp/model

In [ ]:
import sys
import os

# 경로 설정 (clone 받은 폴더 위치)
os.chdir('/content/TensorRT-LLM/examples/llama')

# 체크포인트 변환 및 INT4 양자화 설정
!python convert_checkpoint.py --model_dir /content/tmp/model \
                              --output_dir /content/tmp/tllm_checkpoint \
                              --dtype float16 \
                              --use_weight_only \
                              --weight_only_precision int4_awq

In [ ]:
# 엔진 빌드
!trtllm-build --checkpoint_dir /content/tmp/tllm_checkpoint \
              --output_dir /content/tmp/tllm_engine \
              --gemm_plugin float16

In [ ]:
# 추론 실행
!python ../run.py --engine_dir /content/tmp/tllm_engine \
                  --max_output_len 100 \
                  --tokenizer_dir /content/tmp/model \
                  --input_text "What is the capital of South Korea?"